In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Técnicas de mitigación y supresión de errores

> **Note:** La versión beta de un nuevo modelo de ejecución ya está disponible. El modelo de ejecución dirigida ofrece más flexibilidad para personalizar tu flujo de trabajo de mitigación de errores. Consulta la guía del [Modelo de ejecución dirigida](/guides/directed-execution-model) para más información.


<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Se recomienda usar estas versiones o más recientes.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Las técnicas de mitigación y supresión de errores se utilizan para mejorar la calidad de los resultados al escalar hacia cargas de trabajo más grandes. Esta página proporciona explicaciones de alto nivel de las técnicas de supresión y mitigación de errores disponibles a través de Qiskit Runtime.

La siguiente celda importa el primitivo Estimator y crea un backend que se usará para inicializar el Estimator en celdas de código posteriores.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Desacoplamiento dinámico
Los circuitos cuánticos se ejecutan en hardware de IBM&reg; como secuencias de pulsos de microondas que deben programarse y ejecutarse en intervalos de tiempo precisos.
Desafortunadamente, las interacciones no deseadas entre qubits pueden producir errores coherentes en qubits inactivos. El desacoplamiento dinámico funciona insertando secuencias de pulsos en los qubits inactivos para cancelar aproximadamente el efecto de estos errores. Cada secuencia de pulsos insertada equivale a una operación de identidad, pero la presencia física de los pulsos tiene el efecto de suprimir los errores.
Hay muchas posibles elecciones de secuencias de pulsos, y cuál es mejor para cada caso particular sigue siendo un [área activa de investigación](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

Ten en cuenta que el desacoplamiento dinámico es principalmente útil para circuitos que contienen intervalos en los que algunos qubits permanecen inactivos sin ninguna operación actuando sobre ellos. Si las operaciones del circuito están muy densamente empaquetadas, de modo que todos los qubits están ocupados la mayor parte del tiempo, la adición de pulsos de desacoplamiento dinámico podría no mejorar el rendimiento. De hecho, incluso podría empeorarlo debido a imperfecciones en los propios pulsos.

El diagrama a continuación muestra el desacoplamiento dinámico con una secuencia de pulsos XX. El circuito abstracto de la izquierda se mapea en un calendario de pulsos de microondas en la parte superior derecha. La parte inferior derecha muestra el mismo calendario, pero con una secuencia de dos pulsos X insertados durante un período de inactividad del primer qubit.

![Representación del desacoplamiento dinámico](../docs/images/guides/error-mitigation-explanation/dd.avif)

El desacoplamiento dinámico puede habilitarse estableciendo `enable` en `True` en las [opciones de desacoplamiento dinámico](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). La opción `sequence_type` puede usarse para elegir entre varias secuencias de pulsos diferentes. El tipo de secuencia predeterminado es `"XX"`.

La siguiente celda de código muestra cómo habilitar el desacoplamiento dinámico para el Estimator y elegir una secuencia de desacoplamiento dinámico.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Twirling de Pauli
El twirling, también conocido como [compilación aleatoria](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), es una técnica ampliamente utilizada para convertir canales de ruido arbitrarios en canales de ruido con una estructura más específica.

El twirling de Pauli es un tipo especial de twirling que usa operaciones de Pauli. Tiene el efecto de transformar cualquier canal cuántico en un canal de Pauli. Realizado por sí solo, puede mitigar el ruido coherente porque este tiende a acumularse de forma cuadrática con el número de operaciones, mientras que el ruido de Pauli se acumula de forma lineal. El twirling de Pauli se combina frecuentemente con otras técnicas de mitigación de errores que funcionan mejor con ruido de Pauli que con ruido arbitrario.

El twirling de Pauli se implementa rodeando un conjunto elegido de puertas con puertas de Pauli de un solo qubit elegidas aleatoriamente, de tal manera que el efecto ideal de la puerta permanece igual. El resultado es que un único circuito se reemplaza por un conjunto aleatorio de circuitos, todos con el mismo efecto ideal. Al muestrear el circuito, las muestras se toman de múltiples instancias aleatorias, en lugar de una sola.

![Representación del twirling de Pauli](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

Dado que la mayoría de los errores en el hardware cuántico actual provienen de puertas de dos qubits, esta técnica se aplica frecuentemente de forma exclusiva a puertas de dos qubits (nativas). El siguiente diagrama muestra algunos twirls de Pauli para las puertas CNOT y ECR. Cada circuito dentro de una fila tiene el mismo efecto ideal.

![Representación de los twirls de puertas](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

El twirling de Pauli puede habilitarse estableciendo `enable_gates` en `True` en las [opciones de twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). Otras opciones destacables incluyen:

- `num_randomizations`: El número de instancias de circuito a extraer del conjunto de circuitos con twirling.
- `shots_per_randomization`: El número de disparos a muestrear de cada instancia de circuito.

La siguiente celda de código muestra cómo habilitar el twirling de Pauli y configurar estas opciones para el estimator. Ninguna de estas opciones necesita establecerse explícitamente.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Extinción de errores de lectura con twirling (TREX)
La [extinción de errores de lectura con twirling (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) mitiga el efecto de los errores de medición en la estimación de valores esperados de observables de Pauli.
Se basa en la noción de mediciones con twirling, que se realizan sustituyendo aleatoriamente las puertas de medición por una secuencia de (1) una puerta Pauli X, (2) una medición y (3) un volteo de bit clásico. Al igual que en el twirling de puertas estándar, esta secuencia equivale a una medición simple en ausencia de ruido, como se muestra en el siguiente diagrama:

![Representación del twirling de medición](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

En presencia de errores de lectura, el twirling de medición tiene el efecto de diagonalizar la matriz de transferencia de errores de lectura, facilitando su inversión. Estimar la matriz de transferencia de errores de lectura requiere ejecutar circuitos de calibración adicionales, lo que introduce una pequeña sobrecarga.

TREX puede habilitarse estableciendo `measure_mitigation` en `True` en las [opciones de resiliencia de Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) para el Estimator. Las opciones para el aprendizaje de ruido de medición se describen [aquí](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). Al igual que con el twirling de puertas, puedes establecer el número de aleatorizaciones del circuito y el número de disparos por aleatorización.

La siguiente celda de código muestra cómo habilitar TREX y configurar estas opciones para el estimator. Ninguna de estas opciones necesita establecerse explícitamente.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Extrapolación a ruido cero (ZNE)
La extrapolación a ruido cero (ZNE) es una técnica para mitigar errores en la estimación de valores esperados de observables. Aunque a menudo mejora los resultados, no garantiza producir un resultado sin sesgo.

ZNE consta de dos etapas:

1. _Amplificación de ruido_: El circuito cuántico original se ejecuta múltiples veces a diferentes tasas de ruido.
2. _Extrapolación_: El resultado ideal se estima extrapolando los resultados del valor esperado con ruido hacia el límite de ruido cero.

Tanto la etapa de amplificación de ruido como la de extrapolación pueden implementarse de muchas formas diferentes. Qiskit Runtime implementa la amplificación de ruido mediante "plegado digital de puertas", lo que significa que las puertas de dos qubits se reemplazan por secuencias equivalentes de la puerta y su inversa. Por ejemplo, reemplazar un unitario $U$ con $U U^\dagger U$ produciría un factor de amplificación de ruido de 3. Para la extrapolación, puedes elegir entre varias formas funcionales, incluyendo un ajuste lineal o un ajuste exponencial.
La imagen a continuación muestra el plegado digital de puertas a la izquierda y el procedimiento de extrapolación a la derecha.

![Representación de ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

ZNE puede habilitarse estableciendo `zne_mitigation` en `True` en las [opciones de resiliencia de Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) para el Estimator.
Las opciones de Qiskit Runtime para ZNE se describen [aquí](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Las siguientes opciones son destacables:

- `noise_factors`: Los factores de ruido a usar para la amplificación de ruido.
- `extrapolator`: La forma funcional a usar para la extrapolación.

La siguiente celda de código muestra cómo habilitar ZNE y configurar estas opciones para el estimator. Ninguna de estas opciones necesita establecerse explícitamente.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Amplificación probabilística de errores (PEA)
Uno de los principales desafíos en ZNE es amplificar con precisión el ruido que afecta al circuito objetivo. El plegado de puertas proporciona una manera fácil de realizar esta amplificación, pero puede ser imprecisa y podría llevar a resultados incorrectos. Consulta el artículo ["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197), y específicamente la página 4 de la información suplementaria, para más detalles. La amplificación probabilística de errores ofrece un enfoque más preciso para la amplificación de errores mediante el aprendizaje de ruido.

PEA es una técnica más sofisticada que realiza experimentos preliminares para reconstruir el ruido y luego usa esta información para realizar una amplificación precisa. Comienza aprendiendo el modelo de ruido con twirling de cada capa de puertas de entrelazamiento del circuito antes de ejecutarlas (consulta [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) para las opciones de aprendizaje relevantes). Tras la fase de aprendizaje, los circuitos se ejecutan a cada factor de ruido, donde cada capa de entrelazamiento de los circuitos se amplifica inyectando probabilísticamente ruido de un solo qubit proporcional al modelo de ruido aprendido correspondiente. Consulta el artículo ["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) para más detalles.

PEA consta de tres etapas:
1. _Aprendizaje_: Se aprende el modelo de ruido con twirling de cada capa de puertas de entrelazamiento del circuito.
1. _Amplificación de ruido_: El circuito cuántico original se ejecuta múltiples veces a diferentes factores de ruido.
2. _Extrapolación_: El resultado ideal se estima extrapolando los resultados del valor esperado con ruido hacia el límite de ruido cero.

Para experimentos a escala de utilidad, PEA suele ser la mejor opción.

Dado que PEA es una técnica de amplificación de ruido de ZNE, también debes habilitar ZNE estableciendo `resilience.zne_mitigation = True`. Además, se pueden usar otras opciones de [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) para configurar extrapoladores, niveles de amplificación, etc. PEA requiere un modelo de ruido, que se genera automáticamente al usar primitivos.

El siguiente fragmento proporciona un ejemplo donde se usa PEA para mitigar el resultado de un trabajo del Estimator:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Cancelación probabilística de errores (PEC)
La cancelación probabilística de errores (PEC) es una técnica para mitigar errores en la estimación de valores esperados de observables. A diferencia de ZNE, devuelve una estimación imparcial del valor esperado. Sin embargo, generalmente conlleva una mayor sobrecarga.

En PEC, el efecto de un circuito objetivo ideal se expresa como una combinación lineal de circuitos con ruido que son realmente implementables en la práctica:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

La salida del circuito ideal puede entonces reproducirse ejecutando diferentes instancias de circuitos con ruido extraídas de un conjunto aleatorio definido por la combinación lineal. Si los coeficientes $\eta_i$ forman una distribución de probabilidad, pueden usarse directamente como las probabilidades del conjunto. En la práctica, algunos de los coeficientes son negativos, por lo que forman una distribución de cuasi-probabilidad. Aún pueden usarse para definir un conjunto aleatorio, pero existe una sobrecarga de muestreo relacionada con la negatividad de la distribución de cuasi-probabilidad, caracterizada por la cantidad

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

La sobrecarga de muestreo es un factor multiplicativo sobre el número de disparos requeridos para estimar un valor esperado a una precisión dada, en comparación con el número de disparos que se necesitaría del circuito ideal. Escala cuadráticamente con $\gamma$, que a su vez escala exponencialmente con la profundidad del circuito.

PEC puede habilitarse estableciendo `pec_mitigation` en `True` en las [opciones de resiliencia de Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) para el Estimator.
Las opciones de Qiskit Runtime para PEC se describen [aquí](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). Se puede establecer un límite en la sobrecarga de muestreo usando la opción `max_overhead`. Ten en cuenta que limitar la sobrecarga de muestreo puede hacer que la precisión del resultado supere la precisión solicitada. El valor predeterminado de `max_overhead` es 100.

La siguiente celda de código muestra cómo habilitar PEC y configurar la opción `max_overhead` para el estimator.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Próximos pasos
- Consulta el [tutorial](/tutorials/combine-error-mitigation-techniques) sobre cómo combinar opciones de mitigación de errores con el primitivo Estimator.
- [Configura la mitigación de errores.](configure-error-mitigation)
- [Configura la supresión de errores.](configure-error-suppression)
- Explora otras [opciones](runtime-options-overview) para los primitivos de Qiskit Runtime.
- Decide en qué [modo de ejecución](execution-modes) ejecutar tu trabajo.